In [1]:
from dotenv import load_dotenv
from sqlalchemy import create_engine
import os
import pandas as pd
import numpy as np


load_dotenv()
##Connect to sql database
engine = create_engine(
    f"mysql+pymysql://{os.getenv('MYSQL_USER')}:{os.getenv('MYSQL_PASSWORD')}@"
    f"{os.getenv('MYSQL_HOST')}:{os.getenv('MYSQL_PORT')}/{os.getenv('MYSQL_DB')}"
)

data = pd.read_sql("SELECT * FROM GDP_inference_clean", engine)
print(data)

          date  PRS85006091  UNRATE    DFF  CORESTICKM159SFRBATL        gdp  \
0   1985-01-01          1.4     7.3   8.74              4.714410   4230.168   
1   1985-04-01          1.2     7.3   8.83              4.742850   4294.887   
2   1985-07-01          2.0     7.4   8.12              4.934458   4386.773   
3   1985-10-01          2.3     7.1   8.26              4.877351   4444.094   
4   1986-01-01          3.2     6.7  13.46              5.280953   4507.894   
..         ...          ...     ...    ...                   ...        ...   
135 2018-10-01          0.5     3.8   2.18              2.415236  20917.867   
136 2019-01-01          1.0     4.0   2.40              2.375054  21111.600   
137 2019-04-01          1.6     3.7   2.41              2.392773  21397.938   
138 2019-07-01          2.4     3.7   2.39              2.475923  21717.171   
139 2019-10-01          3.6     3.6   1.88              2.747596  21933.217   

       POPTHM    GPDIC1   FGEXPND      PCE   NETEXP

In [2]:
def renaming(data):
    data.columns = ["date", "Labor Productivity", "Unemployment Rate", "Federal Funds Rate", "Inflation", "GDP",
                    "Population", "Investment", "Government Spending","Consumption","Net Exports"] ##Renaming the columns
    return data

data=renaming(data)
print(data)

          date  Labor Productivity  Unemployment Rate  Federal Funds Rate  \
0   1985-01-01                 1.4                7.3                8.74   
1   1985-04-01                 1.2                7.3                8.83   
2   1985-07-01                 2.0                7.4                8.12   
3   1985-10-01                 2.3                7.1                8.26   
4   1986-01-01                 3.2                6.7               13.46   
..         ...                 ...                ...                 ...   
135 2018-10-01                 0.5                3.8                2.18   
136 2019-01-01                 1.0                4.0                2.40   
137 2019-04-01                 1.6                3.7                2.41   
138 2019-07-01                 2.4                3.7                2.39   
139 2019-10-01                 3.6                3.6                1.88   

     Inflation        GDP  Population  Investment  Government Spending  \
0

In [3]:
def connverting_to_log(data) :
    cols_to_log = ['GDP', 'Investment', 'Consumption', 'Government Spending']

    for i in cols_to_log:
        data[i]=np.log(data[i])

    return data

data=connverting_to_log(data)
print(data)

          date  Labor Productivity  Unemployment Rate  Federal Funds Rate  \
0   1985-01-01                 1.4                7.3                8.74   
1   1985-04-01                 1.2                7.3                8.83   
2   1985-07-01                 2.0                7.4                8.12   
3   1985-10-01                 2.3                7.1                8.26   
4   1986-01-01                 3.2                6.7               13.46   
..         ...                 ...                ...                 ...   
135 2018-10-01                 0.5                3.8                2.18   
136 2019-01-01                 1.0                4.0                2.40   
137 2019-04-01                 1.6                3.7                2.41   
138 2019-07-01                 2.4                3.7                2.39   
139 2019-10-01                 3.6                3.6                1.88   

     Inflation       GDP  Population  Investment  Government Spending  \
0 

In [4]:
def exclusion_reordering(data):
    """
    USE: Excludes unnecessary columns and reorders data for choletsky decomposition

    Paramaters:
    -----------
    data: pd.DataFrame
        Data that you would like to exclude and reorder

    Returns:
    ---------
    data_01_order: pd.DataFrame
        Data with excluded and reordered columns

    """
    exclude = ["date", "GDP_per_capita", "Population","Net Exports"]
    data_i1 = data.loc[:, ~data.columns.isin(exclude)]

    re_order_cols = [
        "Labor Productivity",
        "Government Spending",
        "GDP",
        "Consumption",
        "Investment",
        "Unemployment Rate",
        "Inflation",
        "Federal Funds Rate"
    ]
    ##Reorder columns for choletsky decomposition
    data_i1_reorder = data_i1[re_order_cols]
    data_i1_reorder.index= data["date"]

    return data_i1_reorder

data=exclusion_reordering(data)
print(data)

            Labor Productivity  Government Spending       GDP  Consumption  \
date                                                                         
1985-01-01                 1.4             6.858666  8.349997     7.870471   
1985-04-01                 1.2             6.873876  8.365181     7.885893   
1985-07-01                 2.0             6.892306  8.386349     7.906805   
1985-10-01                 2.3             6.904179  8.399331     7.921463   
1986-01-01                 3.2             6.915182  8.413585     7.947007   
...                        ...                  ...       ...          ...   
2018-10-01                 0.5             8.432377  9.948359     9.552731   
2019-01-01                 1.0             8.452761  9.957578     9.552837   
2019-04-01                 1.6             8.462726  9.971050     9.567315   
2019-07-01                 2.4             8.470786  9.985859     9.581628   
2019-10-01                 3.6             8.475595  9.995758   

In [5]:
dates = pd.date_range(start="1985-01-01", end="2019-12-31", freq='QS') ##The entire period
dummy_variables = pd.DataFrame(index=dates)

def add_dummies(df, break_dates=["1986-04-01","1986-07-01","1987-04-01","1987-07-01","2008-07", "2008-10"]):
    """
    USE: Adds dummy variables for specific break dates

    Paramaters:
    -----------
    df: pd.DataFrame
        Dataframe where you would like to add dummy variables
    break_dates: list
        List of break dates

    Returns:
    ---------
    df: pd.DataFrame
        Dataframes with added dummy variables
    """
    # Ensure index is datetime
    df.index = pd.to_datetime(df.index)

    for bd in break_dates:
        col_name = f"dummy_{bd.replace('-', '_')}" ##Specify column names
        # Compare year and month only
        target = pd.to_datetime(bd) ##convert date to datetime
        df[col_name] = ((df.index.year == target.year) & (df.index.month == target.month)).astype(int) ##Add a dummy variable to specific date and month

    return df

dummy_variables = add_dummies(dummy_variables)
print(dummy_variables)


            dummy_1986_04_01  dummy_1986_07_01  dummy_1987_04_01  \
1985-01-01                 0                 0                 0   
1985-04-01                 0                 0                 0   
1985-07-01                 0                 0                 0   
1985-10-01                 0                 0                 0   
1986-01-01                 0                 0                 0   
...                      ...               ...               ...   
2018-10-01                 0                 0                 0   
2019-01-01                 0                 0                 0   
2019-04-01                 0                 0                 0   
2019-07-01                 0                 0                 0   
2019-10-01                 0                 0                 0   

            dummy_1987_07_01  dummy_2008_07  dummy_2008_10  
1985-01-01                 0              0              0  
1985-04-01                 0              0              0  
